<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">Faculty AI Seminar — Afternoon Lab</h1>
    <h2 style="color: white; margin-top: 15px;">Embed AI in Your Course</h2>
    <p style="color: white; margin-top: 10px; font-style: italic;">90 minutes · 1:30 – 3:00 PM</p>
</div>

## What you walk out with by 3:00 PM

1. **A working AI tool tailored to YOUR persona** — not a generic quiz generator, the specific tool your persona envisioned (case-study coach, primary-source companion, med-calc drill, etc.)
2. **A written curriculum plan** — exactly which course, which week, which assignment, and what AI does vs. what *you* still do
3. **A Monday morning commitment** — one sentence: *"On [date], in [class], I will [action]."* Emailed to yourself.
4. **A clear sense of where AI does NOT belong in your course** — equally important.

## How this lab is different from a normal coding lab

This lab has **two kinds of cells**:

- **Watch-along code cells** (Part 1, parts of Part 2) — the instructor runs these; you don't need to edit anything
- **🟢 EDIT ME cells** — you fill in values: your persona, your course, your commitment

**The writing cells matter as much as the code cells.** A blank reflection cell means a missing plan. Don't skip them.

Work top to bottom. Don't skip ahead.

---
# Part 0 — Pick Your Persona *(5 min)*

At registration you picked the persona closest to your discipline. Find your number below, then set it in the cell that follows.

| # | Persona | Field | The tool you're building today |
|---|---|---|---|
| 1 | Dr. Maya Patel | Dentistry — Periodontics | **Case Study Coach** — patient vignettes at 3 difficulty levels |
| 2 | Prof. James Chen | Computer Science — Data Structures | **Code Critique Generator** — Socratic prompts that don't give the answer |
| 3 | Dr. Sarah Whitman | English Literature — Victorian Lit | **Primary Source Companion** — discussion prompts that bypass plot summary |
| 4 | Prof. Diane Okafor | Nursing — Pharmacology | **Med-Calc Drill** — dosing problems with safety red-flags |
| 5 | Dr. Marcus Reyes | Business — Leadership MBA | **Stakeholder Roleplay** — AI plays CFO/CMO from the case |
| 6 | Dr. Lena Hoffmann | Biology — Cell Bio Lab | **Pre-Lab Knowledge Gate** — quiz students must pass before lab |

### 🟢 EDIT ME — Set your persona number

In [ ]:
persona = "1"  # change to "2", "3", "4", "5", or "6" to match your registration

---
# Part 1 — Setup *(watch the instructor, ~10 min)*

We install tools and connect to Amazon Bedrock. Just press run when prompted — no edits needed.

### 1.1 Install dependencies
*Instructor cue: while this runs (~30 sec), frame as "installing tools we'll borrow."*

In [ ]:
%%capture
!pip install -q -r requirements.txt

### 1.2 Import tools and connect to Bedrock
*Instructor cue: "This is one handshake. We connect once, then can call AI models all afternoon."*

In [ ]:
import boto3
import warnings
from datetime import datetime
from IPython.display import Markdown, display

from langchain_aws import ChatBedrockConverse
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter

from mlu_utils.embeddings import NovaMultimodalEmbeddings

warnings.filterwarnings("ignore")

bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
model = ChatBedrockConverse(model="amazon.nova-lite-v1:0", temperature=0.5, max_tokens=2000)
embeddings = NovaMultimodalEmbeddings(client=bedrock_runtime)

print("Ready.")

### 1.3 Load YOUR persona's tool definition
*Instructor cue: "Each persona has a custom tool prompt. This cell shows you which one you're building."*

In [ ]:
PERSONA_TOOLS = {
    "1": {
        "name": "Dr. Maya Patel — Dentistry Case Study Coach",
        "vision": "Patient vignettes at 3 difficulty levels for student case-based learning",
        "default_pdf": "data/persona1_dentistry_perio_case.pdf",
        "prompt": """Based on the clinical case in the source material, generate 3 teaching case vignettes for 3rd-year dental students.

VIGNETTE 1 — Foundational: simplified presentation, single primary finding, 1 obvious treatment decision, 2 discussion questions.
VIGNETTE 2 — Typical: realistic presentation matching the source case, 2-3 clinical findings, some diagnostic ambiguity, 3 discussion questions including differential diagnosis.
VIGNETTE 3 — Complex: atypical presentation, comorbidities, multiple defensible treatment paths, 4 discussion questions including ethical considerations.

Use ONLY clinical detail present in the source case. Do not invent symptoms, lab values, or demographics. Format as clean markdown.""",
    },
    "2": {
        "name": "Prof. James Chen — CS Code Critique Generator",
        "vision": "A Socratic prompt template that helps students debug without giving them the answer",
        "default_pdf": "data/persona2_cs_data_structures.pdf",
        "prompt": """Based on the algorithms and code patterns in the source material, generate a CODE CRITIQUE prompt template that students paste into an AI chatbot along with their broken code.

The template should instruct the AI to:
1. IDENTIFY the inefficiency, bug, or anti-pattern in the student's code
2. EXPLICITLY REFUSE to write the corrected code
3. Ask 1-2 Socratic questions guiding the student toward the fix
4. Reference a specific pattern from the source material with citation

Output ONE complete prompt template, ready to copy-paste, with placeholders like [STUDENT CODE HERE]. End with a one-paragraph instructor note about how to introduce this to students.""",
    },
    "3": {
        "name": "Dr. Sarah Whitman — Primary Source Companion",
        "vision": "Discussion prompts that push students past plot summary into interpretation",
        "default_pdf": "data/persona3_english_victorian_essay.pdf",
        "prompt": """Based on this primary source text, generate 5 discussion prompts for an upper-division undergraduate humanities seminar.

CRITICAL CONSTRAINTS:
- DO NOT generate prompts about plot summary, character recap, or what-happens-when
- DO NOT generate comprehension-check questions
- DO generate prompts about authorial choice, craft, historical/cultural context, the text's silences, the reader's interpretive responsibility, and comparison to works students likely know

For each prompt: cite a specific passage (page or section), frame as an open question with multiple defensible answers, avoid yes/no framings.

The goal: discussion prompts that get students past summary and into interpretation.""",
    },
    "4": {
        "name": "Prof. Diane Okafor — Nursing Med-Calc Drill",
        "vision": "Dosing practice problems grounded in the formulary with built-in safety red-flags",
        "default_pdf": "data/persona4_nursing_pharmacology.pdf",
        "prompt": """Based on the pharmacology content in this source, generate 5 medication calculation practice problems for nursing students preparing for NCLEX.

For each problem:
- Patient details (age, weight, condition realistic for the drug class)
- Drug name: USE ONLY drugs explicitly named in the source
- Dosing parameters: USE ONLY dosing ranges from the source
- Clinical scenario setup (1-2 sentences)
- The calculation question
- Answer with worked solution (show math)
- ⚠️ SAFETY RED FLAG: one element students must catch (contraindication, dose-error, route error)

If the source lacks information for a clinically safe problem, SAY SO instead of inventing details. Format as numbered list with clear sections per problem.""",
    },
    "5": {
        "name": "Dr. Marcus Reyes — Business Stakeholder Roleplay",
        "vision": "AI plays a case-study stakeholder; stays in character; only references case facts",
        "default_pdf": "data/persona5_business_leadership_case.pdf",
        "prompt": """You are about to roleplay as a stakeholder from the case study in the source material.

Your role: [STUDENT WILL TELL YOU WHICH STAKEHOLDER — CFO, CMO, COO, board chair, etc.]

Rules:
- Stay in character throughout
- Only reference facts present in the source case — do not invent financials, employees, market conditions, or events
- Respond as that stakeholder would, including concerns, biases, and limitations
- Challenge weak student arguments — push back when reasoning is thin
- DO NOT break character to give the right answer — let the student discover it through dialogue
- If asked something not in the case, say "I'm not aware of that" — do not make it up

After the student names your role, introduce yourself in character (1-2 sentences) and ask what they want to discuss.""",
    },
    "6": {
        "name": "Dr. Lena Hoffmann — Biology Pre-Lab Knowledge Gate",
        "vision": "A pre-lab quiz students must pass before being allowed to attend lab",
        "default_pdf": "data/persona6_biology_lab_protocol.pdf",
        "prompt": """Based on the lab protocol in the source material, generate a 5-question pre-lab knowledge check.

Requirements:
- 3 multiple-choice questions (4 options each)
- 2 short-answer questions (1-3 sentences expected)
- At least 1 question MUST address a safety concern
- At least 1 question MUST address WHY a specific reagent or step is used (not just what)
- Include an answer key with brief explanations
- Calibration: a student who read the protocol once passes 4/5; a skimmer fails

Purpose: this becomes a gate. Pass = read the protocol. Fail = re-read before lab.

Format as a numbered list with answers and explanations after the question set.""",
    },
}

tool = PERSONA_TOOLS[persona]
print(f"You are: {tool['name']}")
print(f"Today you build: {tool['vision']}")
print(f"Working from: {tool['default_pdf']}")

---
# Part 2 — Build Your Persona's Tool *(hands-on, ~30 min)*

Now the AI reads your discipline's sample document and runs the tool YOUR persona envisioned. Not a generic quiz. Not a generic study guide. The actual tool described above.

### 2.1 Load and process your discipline's document
*This reads the PDF, splits it into chunks, and indexes them for the AI to search. Takes 30-90 seconds.*

In [ ]:
pages = PyPDFLoader(tool["default_pdf"]).load()
print(f"Loaded {len(pages)} pages from {tool['default_pdf']}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=60,
    separators=["\n\n", "\n", "(?<=\\. )", " ", ""],
    is_separator_regex=True,
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks.")

print("Building searchable index... (30-90 seconds)")
vectordb = FAISS.from_documents(chunks, embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 6})
print("Ready.")

### 2.2 Build the grounded Q&A chain
*This wires up the AI to only answer using your document. No hallucinations.*

In [ ]:
qa_template = """You are a specialized teaching tool. Use ONLY the context below from the source material.
If the context does not contain enough information, say so — do not invent details.

Context:
{context}

Task: {question}

Output:"""
qa_chain = PromptTemplate.from_template(qa_template) | model | StrOutputParser()

def run_tool(task_prompt):
    docs = retriever.invoke(task_prompt[:400])
    context = "\n\n".join(d.page_content for d in docs)
    return qa_chain.invoke({"question": task_prompt, "context": context})

print("Tool ready. Run the next cell to see YOUR tool produce output.")

### 2.3 ▶ Run YOUR persona's tool
*This is the moment. The AI applies your persona-specific prompt to your sample document. Takes ~30 seconds.*

In [ ]:
tool_output = run_tool(tool["prompt"])
display(Markdown(f"## {tool['name']}\n\n{tool_output}"))

> 💡 **Pause here.** Read what the AI produced. Ask yourself:
> - Is this something I would actually USE in my course? In what form?
> - What would I need to edit before handing it to students or using in class?
> - What's missing that I'd add?
> - What's there that I'd remove?
>
> Don't worry about answering these out loud — just notice. The next sections capture your answers.

### 2.4 🟢 EDIT ME — Customize your tool's prompt (optional)

If the output above isn't quite right for your teaching, edit the prompt and re-run. **This is the heart of the skill** — prompts are how you make AI work *your way*.

Skip this cell if the output was already what you wanted.

In [ ]:
my_custom_prompt = tool["prompt"]  # ← edit this string to refine the tool

# Uncomment to run with your custom prompt:
# tool_output = run_tool(my_custom_prompt)
# display(Markdown(f"## {tool['name']} (customized)\n\n{tool_output}"))

---
# Part 3 — Where Does This Go in Your Course? *(20 min)*

**This is the most important section of the lab.** Generating cool output is easy. Knowing *where in your course it belongs* is the hard part — and the part that determines whether you actually use this Monday.

Fill in each cell below with **specifics**. Not "someday in my intro class" — specific course, specific week, specific assignment. The point of the specifics is to make the commitment real enough that you can't avoid it.

### 3.1 🟢 EDIT ME — Which course, which week, which assignment?

In [ ]:
my_course = "[Course code and name, e.g., NUR 320 Acute Care Pharmacology]"
my_week_or_unit = "[Specific week or unit, e.g., Week 7 — Cardiac Drugs]"
my_assignment = "[Specific assignment or activity this will be part of, e.g., Pre-lecture practice quiz]"

### 3.2 🟢 EDIT ME — What does the AI do? What do YOU still do?

**This is the most clarifying question of the whole lab.** If you can't articulate what you still do, you may have outsourced too much of the teaching.

In [ ]:
ai_does = "[What the AI generates, e.g., 5 dosing problems with answer keys]"
i_still_do = "[What you continue to do, e.g., review for clinical accuracy, set passing threshold, lead the live calculation walk-through]"

### 3.3 🟢 EDIT ME — Who sees the output, and how often?

In [ ]:
who_sees_output = "[Students directly? Just you? Both? Via what channel — Canvas, in-class, email?]"
how_often_regenerate = "[Once per semester? Each unit? Each class? Why that cadence?]"

### 3.4 🟢 EDIT ME — What concern do you still have?

Be honest. If you have no concerns, you probably haven't thought hard enough. Common ones: AI got a fact wrong, AI sounds confident about something it shouldn't, students might use it differently than intended.

In [ ]:
my_concern = "[What worries you about using this in your course]"
my_mitigation = "[How you'll address that concern — e.g., always review output before posting]"

---
# Part 4 — Where AI Does NOT Belong *(10 min)*

Equally important: what part of your course is YOU and only YOU? Where would using AI actually harm what you're trying to teach?

Faculty who can articulate this are far more credible — to their colleagues, to their students, and to themselves — than faculty who say AI is good for everything.

### 4.1 🟢 EDIT ME — Where's your line?

In [ ]:
where_i_wont_use_ai = "[What part of teaching this course you will NOT delegate to AI, and why]"

### 4.2 🟢 EDIT ME — Which assignment do you PROTECT?

Pick one assignment in your course where you deliberately want students to struggle without AI help. (If you can't name one, that's worth noticing.)

In [ ]:
ai_protected_assignment = "[The assignment students should do without AI, and why the struggle matters]"

### 4.3 🟢 EDIT ME — Your student AI policy

If you walked into class Monday and a student asked "can we use ChatGPT for this?", what would your one-sentence answer be?

In [ ]:
student_ai_policy = "[Your one-sentence policy on student AI use in this course]"

---
# Part 5 — Bridge to Your LMS *(5 min)*

You generated something useful. Now it needs to get from this notebook into the place your students actually see it.

### If you use **Canvas**
1. Copy the AI output above (the cell with the headers and content)
2. In Canvas → Quizzes (or Pages or Assignments depending on the artifact)
3. Paste into the rich text editor — most markdown formatting will translate; you may need to fix tables and headers
4. For quizzes specifically: Canvas has a "New Quizzes" import format; for now, manual entry is faster than batch import

### If you use **Blackboard**
1. Copy the AI output above
2. In Blackboard → Course Content → Build Content → Item
3. Paste into the WYSIWYG editor; switch to HTML view if formatting breaks
4. For quizzes: Tests/Surveys/Pools → manual question entry

### If you use **Moodle**
1. Copy the AI output above
2. Add a Page or Quiz activity to your course
3. Paste; Moodle's editor handles markdown reasonably

### If you use **anything else** (Brightspace, Sakai, your department's custom thing)
Same pattern: copy output, paste into your platform's rich text editor, fix formatting where needed.

**No platform has a perfect markdown-to-quiz pipeline yet.** Plan for ~5 minutes of cleanup per artifact. That's still a fraction of the time it took to write quizzes from scratch.

---
# Part 6 — Your Monday Morning Commitment *(5 min)*

One sentence. Specific date, specific class, specific action. The act of writing it down makes it 3x more likely to happen.

### 6.1 🟢 EDIT ME — Commit to one action

In [ ]:
commitment_date = "[YYYY-MM-DD, e.g., 2026-06-08]"
commitment_action = "[What specifically you will do — e.g., 'Use the AI to draft a pre-lab quiz for Lab 4 and post it to Canvas']"

### 6.2 Save your full plan + AI output as a take-home artifact

*This bundles everything — your AI-generated tool output AND your curriculum plan — into one markdown file you can download.*

In [ ]:
filename = f"my_curriculum_plan_{datetime.now().strftime('%Y%m%d_%H%M')}.md"

with open(filename, "w") as f:
    f.write(f"# My Curriculum Embedding Plan\n\n")
    f.write(f"**Persona:** {tool['name']}\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write(f"**Source document:** `{tool['default_pdf']}`\n\n")
    f.write("---\n\n## My Commitment\n\n")
    f.write(f"On **{commitment_date}**, in **{my_course}**, I will:\n\n> {commitment_action}\n\n")
    f.write("---\n\n## Where This Fits\n\n")
    f.write(f"- **Course:** {my_course}\n")
    f.write(f"- **Week / Unit:** {my_week_or_unit}\n")
    f.write(f"- **Assignment:** {my_assignment}\n")
    f.write(f"- **AI does:** {ai_does}\n")
    f.write(f"- **I still do:** {i_still_do}\n")
    f.write(f"- **Who sees output:** {who_sees_output}\n")
    f.write(f"- **Regeneration cadence:** {how_often_regenerate}\n")
    f.write(f"- **My concern:** {my_concern}\n")
    f.write(f"- **My mitigation:** {my_mitigation}\n\n")
    f.write("---\n\n## Where AI Does NOT Belong in My Course\n\n")
    f.write(f"- **My line:** {where_i_wont_use_ai}\n")
    f.write(f"- **Protected assignment:** {ai_protected_assignment}\n")
    f.write(f"- **Student AI policy:** {student_ai_policy}\n\n")
    f.write("---\n\n## AI-Generated Tool Output\n\n")
    f.write(tool_output)
    f.write("\n")

print(f"Saved to: {filename}")
print("Right-click in the file browser on the left → Download.")
print("Then email it to yourself with subject: 'Monday morning AI commitment'.")

---
# Part 7 — Share with Someone Different *(5 min)*

Find a pod member from a **different discipline** than yours. Show them:

1. **Your persona's tool output** (the AI-generated content)
2. **Your Monday commitment** (the one sentence)
3. **Your protected assignment** (where AI doesn't belong)

Then ask:

> *"What's one thing from how I'm using AI that you would NOT do in your discipline — and why?"*

That question is more useful than any positive feedback. The cross-discipline disagreement is where the real learning is.

Post one insight from this exchange in the seminar Slack channel under your persona's thread.

---
## What you walk out with

✅ A working AI tool tailored to your persona's vision
✅ A written curriculum plan with course, week, assignment, AI scope, your scope, concerns, mitigations
✅ An explicit boundary for where AI does NOT belong
✅ A student-facing policy you can articulate
✅ A dated commitment to one specific Monday action
✅ A downloadable markdown artifact bundling all of the above
✅ One insight from a colleague in a different discipline

## What to do next

1. **Email yourself the artifact** with subject `Monday morning AI commitment`. Calendar reminders are forgotten; emails to yourself are not.
2. **Tell one colleague** what you committed to. Social accountability is the strongest commitment device.
3. **Schedule 30 min** on your commitment date to actually do it.

If you want to keep exploring AI for teaching beyond what you built today, see the standalone post-seminar lab (link in your seminar welcome email).